# Xin Ying's Deep Learning Pipeline: PyTorch TabularNN with Entity Embeddings
**Course:** IT2226 - AI Innovation and Deep Learning Project  
**Target:** Predict 30-day repeat purchase in October 2024 (Class 1 = 14.57% positive)  
**Validation Strategy:** Strict 80/20 Train/Test Holdout Split (`seed=42`)

### Key Highlights:
1. **Categorical Entity Embeddings:** Dense vector representations for `country_code`, `customer_tier`, and `preferred_channel`.
2. **Continuous Feature Normalization:** Batch normalization and standard scaling fitted strictly on the 80% training split to eliminate data leakage.
3. **Class Imbalance Handling:** Positive class weighting (`pos_weight = 5.863`) in BCE loss.
4. **Holdout Evaluation:** Evaluated on the held-out 20% test split.

In [ ]:
# 1. Setup & Environment
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    f1_score,
    recall_score,
    precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Set seeds globally for 100% deterministic reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__} | Device: {DEVICE}')

In [ ]:
# 2. Load Customer 360 Feature Matrix
FEATURE_PATH = 'task_outputs/model_feature_matrix.csv'
if not os.path.exists(FEATURE_PATH):
    FEATURE_PATH = '../../task_outputs/model_feature_matrix.csv'

df = pd.read_csv(FEATURE_PATH)
print(f'Total customer records: {len(df):,}')

CAT_COLS = ['country_code', 'customer_tier', 'preferred_channel']
DROP_COLS = ['customer_id', 'target', 'is_test']
NUM_COLS = [c for c in df.columns if c not in CAT_COLS and c not in DROP_COLS]

train_df = df[df['is_test'] == 0].copy().reset_index(drop=True)
test_df = df[df['is_test'] == 1].copy().reset_index(drop=True)

print(f'Train pool samples: {len(train_df):,} | Unlabeled test samples: {len(test_df):,}')
print(f'Categorical features ({len(CAT_COLS)}): {CAT_COLS}')
print(f'Continuous numerical features ({len(NUM_COLS)}): {len(NUM_COLS)} features')

In [ ]:
# 3. Categorical Encodings & Median Imputation
cat_dims = []
cat_encoders = {}

for c in CAT_COLS:
    train_df[c] = train_df[c].fillna('Unknown').astype(str)
    test_df[c] = test_df[c].fillna('Unknown').astype(str)
    le = LabelEncoder()
    train_df[c] = le.fit_transform(train_df[c])
    test_df[c] = le.transform(test_df[c])
    cat_encoders[c] = le
    n_classes = len(le.classes_)
    emb_dim = max(4, (n_classes + 1) // 2)
    cat_dims.append((n_classes, emb_dim))

for col in NUM_COLS:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med)
    test_df[col] = test_df[col].fillna(med)

y = train_df['target'].astype(int).values
print(f'Dataset positive rate: {y.mean():.2%}')

In [ ]:
# 4. PyTorch Dataset and TabularNN Architecture
class TabularDataset(Dataset):
    def __init__(self, cat_data, num_data, targets=None):
        self.cat_data = torch.tensor(cat_data, dtype=torch.long)
        self.num_data = torch.tensor(num_data, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32) if targets is not None else None

    def __len__(self):
        return len(self.cat_data)

    def __getitem__(self, idx):
        if self.targets is not None:
            return self.cat_data[idx], self.num_data[idx], self.targets[idx]
        return self.cat_data[idx], self.num_data[idx]

class TabularNN(nn.Module):
    def __init__(self, emb_dims, num_numerical, hidden_dims=[128, 64, 32], dropout_p=0.25):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(num_c, dim) for num_c, dim in emb_dims])
        total_emb_dim = sum(dim for _, dim in emb_dims)
        self.num_bn = nn.BatchNorm1d(num_numerical)
        
        layers = []
        in_dim = total_emb_dim + num_numerical
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_p))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        emb_outs = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embeddings)]
        x_emb = torch.cat(emb_outs, dim=1)
        x_num_norm = self.num_bn(x_num)
        x = torch.cat([x_emb, x_num_norm], dim=1)
        return self.mlp(x).squeeze(1)

print('TabularNN Model Architecture initialized.')

In [ ]:
# 5. 80/20 Stratified Train/Test Split & Leakage-Free Scaling
X_cat_all = train_df[CAT_COLS].values
X_num_all = train_df[NUM_COLS].values

X_cat_tr, X_cat_te, X_num_tr_raw, X_num_te_raw, y_tr, y_te = train_test_split(
    X_cat_all, X_num_all, y, test_size=0.20, stratify=y, random_state=SEED
)

print(f'Training Set (80%): {len(y_tr):,} samples (Positives: {y_tr.sum():,})')
print(f'Holdout Test (20%): {len(y_te):,} samples (Positives: {y_te.sum():,})')

# Fit scaler exclusively on the 80% training set
scaler = StandardScaler()
X_num_tr = scaler.fit_transform(X_num_tr_raw)
X_num_te = scaler.transform(X_num_te_raw)

# Loss function with positive class weighting
pos_weight_tr = (len(y_tr) - y_tr.sum()) / y_tr.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_tr], device=DEVICE))

train_loader = DataLoader(TabularDataset(X_cat_tr, X_num_tr, y_tr), batch_size=512, shuffle=True)
test_loader = DataLoader(TabularDataset(X_cat_te, X_num_te, y_te), batch_size=512, shuffle=False)

model = TabularNN(cat_dims, len(NUM_COLS)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

print(f'Training on 80% holdout with pos_weight={pos_weight_tr:.3f} ...')

In [ ]:
# 6. Model Training Loop (12 Epochs)
EPOCHS = 12
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for bc, bn, bt in train_loader:
        bc, bn, bt = bc.to(DEVICE), bn.to(DEVICE), bt.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(bc, bn), bt)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(bt)
    
    scheduler.step()
    epoch_loss = running_loss / len(train_loader.dataset)
    if epoch % 3 == 0 or epoch == EPOCHS:
        print(f'Epoch {epoch:2d}/{EPOCHS} | Train Loss: {epoch_loss:.4f}')

print(f'Training completed in {time.time() - t0:.1f}s.')

In [ ]:
# 7. Evaluation on the 20% Hold-out Test Set
model.eval()
test_preds = []
with torch.no_grad():
    for bc, bn, _ in test_loader:
        bc, bn = bc.to(DEVICE), bn.to(DEVICE)
        probs = torch.sigmoid(model(bc, bn)).cpu().numpy()
        test_preds.extend(probs)

test_probs = np.array(test_preds)
holdout_auc = roc_auc_score(y_te, test_probs)
holdout_pr_auc = average_precision_score(y_te, test_probs)

# Optimal threshold search on F1
best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.2, 0.8, 0.02):
    f1 = f1_score(y_te, (test_probs >= th).astype(int))
    if f1 > best_f1:
        best_f1, best_th = f1, th

y_pred_binary = (test_probs >= best_th).astype(int)
rec = recall_score(y_te, y_pred_binary)
prec = precision_score(y_te, y_pred_binary)
acc = np.mean(y_pred_binary == y_te)

print('=' * 65)
print('FINAL 20% HOLDOUT EVALUATION (PyTorch Tabular DL, seed=42):')
print(f'  ROC-AUC:    {holdout_auc:.4f}')
print(f'  PR-AUC:     {holdout_pr_auc:.4f}')
print(f'  Best F1:    {best_f1:.4f} (at threshold {best_th:.2f})')
print(f'  Recall:     {rec:.4f} ({rec*100:.2f}%)')
print(f'  Precision:  {prec:.4f} ({prec*100:.2f}%)')
print(f'  Accuracy:   {acc:.4f} ({acc*100:.2f}%)')
print('=' * 65)

print('\nClassification Report on 20% Holdout:')
print(classification_report(y_te, y_pred_binary, digits=4))